# Image Embedding (ResNet50) and Correlations with NAcc Activity
**Authors: Brenden Eum & Tom Henning (2024)**

This notebook contains the initial pipeline for taking car images and determining the latent features that explain the most variance in NAcc activity.

---

Inputs:
- Vehicle images in /experiments/stimuli/TRI_car_imgs/.
- Default ResNet50 weights.

Outputs: 
- A sorted list of embedded features that best explain (randomly) simulated NAcc activity.
- A data frame, indexed by image name, with one feature manipulated. Following the Sendhil method. 

---

In [1]:
###########
# Libraries
###########

# General.
import sys
import os
# For images.
from PIL import Image
# For data analysis.
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression #from scikit-learn
# For machine learning.
import torch
from torchvision import transforms, models
import torchvision.transforms as transforms
#from torchvision.models import resnet50
from torchvision.models import ResNet50_Weights
# For visualization.
import matplotlib.pyplot as plt

###########
# Directories
###########

img_dir = "../../experiment/stimuli/Stimuli/TRI_car_imgs"
tmp_dir = "../outputs/temp"
fig_dir = "../outputs/figures"
tab_dir = "../outputs/tables"
txt_dir = "../outputs/text"
fin_dir = "../submitted_figures"
sys.path.append(os.path.abspath("../helpers"))

###########
# Settings
###########

np.random.seed(4)
from myutilities import dict_head # head function for dictionaries

## Vehicle image preprocesssing.

Set up vehicle image preprocessing functions, then load the images and apply the preprocessing functions. Store a dictionary of preprocessed image tensors, keys are image names.

In [2]:
def normalize_img(img_dir, img_name, r=512, g=512, b=512, x=512, y=512):
    """Convert to rgb image. Turn transparent background into white mask. Adjust the image to be X x Y pixels."""
    png_img = Image.open(fr"{img_dir}/{img_name}") #"../../experiment/stimuli/Stimuli/TRI_car_imgs"
    img = Image.new("RGB", png_img.size, (r, g, b)) # Convert to RGB.
    img.paste(png_img, mask=png_img.split()[3])  # Use the alpha channel as a mask. Transparent -> White.
    img = img.resize((x,y)) # Resize to a fixed input size for pre-trained models.
    return(img)
 
def transform_for_resnet(img):
    """Normalize rgb values using ResNet50 statistics, so they're scaled to match ResNet training distributions."""
    preprocess = transforms.Compose([
        transforms.ToTensor(), # Convert to tensor
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    img_tensor = preprocess(img).unsqueeze(0) # Add batch dimension
    return(img_tensor)

def preprocess_img(img_dir, img_name, r=512, g=512, b=512, x=512, y=512):
    """Do all the other functions together."""
    img = normalize_img(img_dir, img_name, r, g, b, x, y)
    img_tensor = transform_for_resnet(img)
    return(img_tensor)

In [3]:
# Specify the folder path
folder_path = "../../experiment/stimuli/Stimuli/TRI_car_imgs/"

# List all files in the folder
all_files = os.listdir(folder_path)

# Filter for PNG images (case-insensitive)
png_files = [file for file in all_files if file.lower().endswith(".png")]

# List of image tensors
img_tensors = {}
for img_name in png_files:
    img_tensors[img_name] = preprocess_img(img_dir, img_name)

dict_head(img_tensors, 1)

{'2021_Toyota_C-HR_XLE_BlackSandPearl_3-4angle_US.png': tensor([[[[2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
           [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
           [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
           ...,
           [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
           [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
           [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489]],
 
          [[2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
           [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
           [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
           ...,
           [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
           [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
           [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286]],
 
          [[2.6400, 2.6400, 2.6400,  ..., 2.6400, 2.6400, 2.6400],
           [2.6400, 2.6400, 2.6400, 

## Get image embeddings.

Run images through model. Store a dictionary of image embeddings, keys are image names.

In [4]:
# Load pre-trained ResNet50.
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2) # the default weights
# Remove the final classification layer to get embeddings.
model = torch.nn.Sequential(*list(model.children())[:-1])
# Set the model to evaulation mode.
model.eval()

# Placeholder dictionary of image embeddings. Fill with the embeddings.
img_embeddings = {}
for tsr in img_tensors.keys():
    with torch.no_grad():
        img_embeddings[tsr] = model(img_tensors[tsr]).squeeze()

dict_head(img_embeddings,4)

{'2021_Toyota_C-HR_XLE_BlackSandPearl_3-4angle_US.png': tensor([0.0000, 0.0021, 0.0000,  ..., 0.1351, 0.0034, 0.0020]),
 '2021_Toyota_Venza_Black_3-4angle_US.png': tensor([0.0424, 0.0065, 0.2235,  ..., 0.3871, 0.0258, 0.0040]),
 '2022_Toyota_Camry_SE_MidnightBlackMetallic_3-4angle_US.png': tensor([0.0180, 0.0404, 0.0453,  ..., 0.5889, 0.0269, 0.0304]),
 '2022_Toyota_CamryHybrid_SE_SuperWhite_3-4angle_US.png': tensor([0.0267, 0.0435, 0.0411,  ..., 0.1664, 0.0100, 0.0154])}

## NAcc activity.

Simulate NAcc activity for now.

In [5]:
NAcc = np.random.normal(loc=0, scale=1, size=72)

## Sorted features.

Regress NAcc activity on the image embeddings. This will generate a list of coefficients. Sort the absolute value of the coefficients in descending order to get a sorted list of the most important embedding features.

In [6]:
# Convert img_embeddings dictionary into a dataframe for regression.
reg_data = pd.DataFrame(img_embeddings).T

# Add a constant to the independent variables df.
X = sm.add_constant(reg_data)

# Fit the linear regression model and extract the coefficients.
model = sm.OLS(NAcc, X).fit()
coefficients = model.params

# Sort coefficients by their absolute values, descending. Turn the feature number into the index.
sorted_coefficients = coefficients.abs().sort_values(ascending=False)
sorted_features = coefficients[sorted_coefficients.index]

sorted_features.head(4)

1318    2.797662
1576    2.559382
464     2.392507
1807   -2.330399
dtype: float64

## Manipulate features.

Set up functions to manipulate one feature at a time using f(feature_vector). Then, do the manipulation.

In [7]:
def manipulation_add(vector, manipulation_args):
    """Add a constant to the variable."""
    return(vector + manipulation_args["constant"])

def manipulate_feature(df, feature, manipulation, manipulation_args):
    """Apply manipulation function to feature at given index, then return new df."""
    mani_embed = df.copy(deep=True)
    mani_embed.iloc[:,feature] = manipulation(mani_embed.iloc[:,feature], manipulation_args)
    return(mani_embed)

In [8]:
# Manipulate the first feature to verify.
feature_idx = 0
feature_sd = np.std(reg_data.iloc[:,feature_idx])
manipulation_args = {"constant":feature_sd*500}

mani_df = manipulate_feature(reg_data, feature_idx, manipulation_add, manipulation_args)
mani_df.head(4)

,0,1,2,3,4,5,6,7,8,9,...,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047
2021_Toyota_C-HR_XLE_BlackSandPearl_3-4angle_US.png,5.177249,0.002147,0.000000,0.000000,0.001131,0.000000,0.002085,0.000000,0.001704,0.000000,...,0.000226,0.036207,0.002017,0.000091,0.000000,0.000000,0.006523,0.135090,0.003421,0.002012
2021_Toyota_Venza_Black_3-4angle_US.png,5.219680,0.006548,0.223481,0.001124,0.003840,0.000051,0.002739,0.013857,0.000000,0.000000,...,0.005709,0.095066,0.018050,0.001336,0.007956,0.000000,0.033041,0.387074,0.025800,0.004037
2022_Toyota_Camry_SE_MidnightBlackMetallic_3-4angle_US.png,5.195252,0.040373,0.045305,0.000000,0.000677,0.000000,0.009174,0.026117,0.000000,0.002101,...,0.001335,0.080794,0.012282,0.007589,0.009076,0.008659,0.042166,0.588891,0.026937,0.030380
2022_Toyota_CamryHybrid_SE_SuperWhite_3-4angle_US.png,5.203975,0.043515,0.041136,0.002267,0.001023,0.000060,0.001944,0.032930,0.000599,0.000000,...,0.000804,0.036820,0.034207,0.013460,0.007982,0.006639,0.021879,0.166401,0.009953,0.015421


In [9]:
# Manipulate the most important feature.
feature_idx = sorted_features.index[0]
feature_sd = np.std(reg_data.iloc[:,feature_idx])
manipulation_args = {"constant":feature_sd}

mani_df = manipulate_feature(reg_data, feature_idx, manipulation_add, manipulation_args)
mani_df.head(4)

,0,1,2,3,4,5,6,7,8,9,...,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047
2021_Toyota_C-HR_XLE_BlackSandPearl_3-4angle_US.png,0.000000,0.002147,0.000000,0.000000,0.001131,0.000000,0.002085,0.000000,0.001704,0.000000,...,0.000226,0.036207,0.002017,0.000091,0.000000,0.000000,0.006523,0.135090,0.003421,0.002012
2021_Toyota_Venza_Black_3-4angle_US.png,0.042430,0.006548,0.223481,0.001124,0.003840,0.000051,0.002739,0.013857,0.000000,0.000000,...,0.005709,0.095066,0.018050,0.001336,0.007956,0.000000,0.033041,0.387074,0.025800,0.004037
2022_Toyota_Camry_SE_MidnightBlackMetallic_3-4angle_US.png,0.018003,0.040373,0.045305,0.000000,0.000677,0.000000,0.009174,0.026117,0.000000,0.002101,...,0.001335,0.080794,0.012282,0.007589,0.009076,0.008659,0.042166,0.588891,0.026937,0.030380
2022_Toyota_CamryHybrid_SE_SuperWhite_3-4angle_US.png,0.026726,0.043515,0.041136,0.002267,0.001023,0.000060,0.001944,0.032930,0.000599,0.000000,...,0.000804,0.036820,0.034207,0.013460,0.007982,0.006639,0.021879,0.166401,0.009953,0.015421


Take one of the rows from mani_df and run that back through generative model. This will allow us to visualize the manipulation for that vehicle.